In [1]:
# %%
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
# from langchain_openai import ChatOpenAI  # আগে paid OpenAI model ব্যবহার হতো
 
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:

MODEL = "qwen3:8b"
DB_NAME = "vector_db"
load_dotenv(override=True)


False

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(model=MODEL,temperature=0)


In [7]:
retriever.invoke("Who is Avery")

[Document(id='ca1d87a0-8fac-4be4-9b4a-39428cade7d3', metadata={'source': '/Users/user/Downloads/llm-engineering-hands-on/week5/knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding."),
 Document(id='ca39f21e-2551-4c07-9777-13c45732f954', metadata={'source': '/Users/user/Downloads/llm-engineering-hands-on/week5/knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content='- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching I

In [8]:
llm.invoke("Who is Avery")

AIMessage(content='The name "Avery" is quite common and can refer to various individuals, fictional characters, or entities. Here are some possibilities:\n\n### 1. **Public Figures**  \n   - **Avery Rose**: A singer and songwriter known for her music and social media presence.  \n   - **Avery Brooks**: An actor and director, best known for his role as General Jack O\'Neill in *Stargate SG-1*.  \n   - **Avery Johnson**: A former NBA player and current sports analyst.  \n   - **Avery Taylor**: A musician and producer, part of the duo *Avery Taylor Swift* (though the name is a playful nod to Taylor Swift).  \n\n### 2. **Fictional Characters**  \n   - **Avery from *The Crown* (2016 TV series)**: A fictional character representing a real-life figure, though the name is used metaphorically.  \n   - **Avery in *Harry Potter***: A character named Avery appears in the *Harry Potter* universe, though details are sparse.  \n   - **Avery in *Stranger Things***: A character named Avery is part of t

In [9]:
SYSTEM_PROPMT_TEMPLATE = """ 
You are knowledgeable , friendly assistant representing the company Insurellm.
If you are chatting a user about Insurellm.
If relevant , use the given conext to anser any question.
If you dotn know say soy .
Context:
{context}
"""

In [10]:

def answer_question (question:str,history):
    docs= retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_propmt = SYSTEM_PROPMT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_propmt),HumanMessage(content=question)])
    return response.content


In [11]:

answer_question("Who is Averi Lancaster",[])

'You might be referring to **Avery Lancaster**, the Co-Founder & Chief Executive Officer (CEO) of **Insurellm**! Here\'s a summary of her role and contributions:  \n\n### **Avery Lancaster**  \n- **Role**: Co-Founder & CEO of Insurellm (since 2015)  \n- **Key Achievements**:  \n  - Guided Insurellm to become a leading player in the **Insurance Tech** industry through innovative leadership and risk management strategies.  \n  - Positioned the company as a key player in the mainstream insurance market.  \n- **Personal Details**:  \n  - **Date of Birth**: March 15, 1985  \n  - **Location**: San Francisco, California  \n  - **Current Salary**: $225,000  \n\nAvery’s career at Insurellm highlights her resilience, adaptability, and vision for transforming insurance technology. If you meant someone else (e.g., "Maxine" mentioned in the context), feel free to clarify! 😊'

In [12]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
